In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk


In [19]:
df = pd.read_csv("Kenya_Coffee_Export_Exchange_Rate.csv")

#checking if df has been loaded
# print(df)

#looking at the heads we are working with
columns = df.columns
print(columns)

#checking how the data looks like in the tables
head = df.head(10)
print(head)

# checking the number non-null values in the columns
df.describe()

# the shape of the data
shape = df.shape
print(shape)

# checking the data types of the columns
dtypes = df.dtypes
print(dtypes)

Index(['year', 'average_kenyan_coffee_price_usd',
       'average_foreign_coffee_price_usd', 'current_total_factor_productivity',
       'germany_import_coffee_price_usd', 'belgium_import_coffee_price_usd',
       'united_states_import_coffee_price_usd',
       'south_korea_import_coffee_price_usd', 'annual_coffee_production',
       'annual_exportable_coffee', 'annual_coffee_exports',
       'rashid_moledina_exports', 'kenya_cooperative_coffee_exporters_exports',
       'dormans_exports', 'mwangi_coffee_exporters_exports',
       'sannex_coffee_exports', 'africoff_trading_exports',
       'diamond_coffee_company_exports', 'rockbern_coffee_group_exports',
       'real_exchange_rate_percent', 'nominal_exchange_rate_percent',
       'average_agricultural_price', 'total_annual_imports',
       'total_annual_exports', 'real_gdp_growth_percent',
       'gross_capital_formation_percent', 'population_growth_rate_percent',
       'real_interest_rate_percent', 'natural_resource_endowment'],
   

In [20]:
# checking for null values in the columns
print(df.isnull().sum())

# drop all unecessary columns that are not needed for the analysis
df = df.drop(columns=[
    "germany_import_coffee_price_usd", "belgium_import_coffee_price_usd", "united_states_import_coffee_price_usd",
    "south_korea_import_coffee_price_usd", "rashid_moledina_exports", "kenya_cooperative_coffee_exporters_exports",
    "dormans_exports", "mwangi_coffee_exporters_exports", "sannex_coffee_exports",
    "africoff_trading_exports", "diamond_coffee_company_exports", "rockbern_coffee_group_exports","nominal_exchange_rate_percent","annual_exportable_coffee"
])

print(df.columns)
print(df.shape)
#  the 8 individual columns were dropped as they are not real independent data every single 
# one of them has a fixed propotion of the tottal exports and applied indetically across all the years. 
# The columns were dropped to avoid multicollinearity in the model and to avoid overfitting the model.

year                                          0
average_kenyan_coffee_price_usd               0
average_foreign_coffee_price_usd              0
current_total_factor_productivity             0
germany_import_coffee_price_usd               0
belgium_import_coffee_price_usd               0
united_states_import_coffee_price_usd         0
south_korea_import_coffee_price_usd           0
annual_coffee_production                      0
annual_exportable_coffee                      0
annual_coffee_exports                         0
rashid_moledina_exports                       0
kenya_cooperative_coffee_exporters_exports    0
dormans_exports                               0
mwangi_coffee_exporters_exports               0
sannex_coffee_exports                         0
africoff_trading_exports                      0
diamond_coffee_company_exports                0
rockbern_coffee_group_exports                 0
real_exchange_rate_percent                    0
nominal_exchange_rate_percent           

## FEATURE ENGINEERING
New columns that will fit the model properly before modelling


In [ ]:
# step 1 : price gap in USD how competitely priced is kenya coffee vs the international market.
# Positive gap → foreign buyers pay more elsewhere → Kenya is cheaper → demand advantage
# Negative gap → Kenya prices its coffee above foreign market → possible demand loss

df['price_gap_usd'] = (
    df['average_foreign_coffee_price_usd'] - df['average_kenyan_coffee_price_usd']
).round(3)
df.head(10)

# in 9 of 20 years kenyay's coffe was priced above the international market price. 
# In 11 of the 20 years kenya's coffee was priced below the international market price. 
# The average price gap is -0.13 USD which means that on average kenya's coffee is priced below the international market price by 0.13 USD.
# meaning kenya coffee commanded a quality premium as the gap turnd positive from 2005 onwards.
# reflecting  the rise in commodity price ad growing coffee demand in the international market.


#STEP 2: THE VOLUME INTERACTIONS
df['price_volume_interaction'] = (df['average_kenyan_coffee_price_usd'] * df['total_annual_exports']).round(3)
df.head(10)

# STEP 3:

,year,average_kenyan_coffee_price_usd,average_foreign_coffee_price_usd,current_total_factor_productivity,annual_coffee_production,annual_coffee_exports,real_exchange_rate_percent,average_agricultural_price,total_annual_imports,total_annual_exports,real_gdp_growth_percent,gross_capital_formation_percent,population_growth_rate_percent,real_interest_rate_percent,natural_resource_endowment,price_gap_usd,price_volume_interaction
0,2001,69.400000,37.05,0.516281,991,1096,147.160209,62.881144,33.015260,122.220962,3.779906,18.790341,2.728049,17.812501,0.161256,-32.350,8482.135
1,2002,67.670000,30.91,0.503528,945,736,172.402278,66.923841,30.274700,136.749966,0.546860,15.138216,2.712397,17.358141,0.155309,-36.760,9253.870
2,2003,41.070000,42.82,0.436189,673,920,72.832177,70.818207,30.045451,144.205080,2.932476,16.482149,2.709585,9.770511,0.152397,1.750,5922.503
3,2004,71.010000,56.33,0.425102,736,754,99.807153,70.080806,32.866745,144.812870,5.104300,16.962496,2.720779,5.045258,0.151127,-14.680,10283.162
4,2005,70.079945,87.09,0.439443,660,673,60.797196,73.237087,35.969836,163.254188,5.906666,17.649685,2.739246,7.609988,0.146863,17.010,11440.844
5,2006,76.904891,87.02,0.384618,826,597,63.719913,82.823877,32.251545,155.698224,6.472494,18.633585,2.757917,-8.009867,0.143804,10.115,11973.955
6,2007,72.209092,98.30,0.391849,652,817,49.450107,85.169692,31.975798,172.922342,6.850730,20.456978,2.768549,4.819091,0.139339,26.091,12486.565
7,2008,71.608141,109.26,0.376927,541,608,45.336958,87.536325,34.904541,183.616577,0.232283,19.612711,2.767256,-0.984997,0.135249,37.652,13148.442
8,2009,70.624260,100.80,0.354472,630,525,54.195720,85.822744,27.170246,164.548811,3.306940,19.019172,2.750854,-10.096004,0.136259,30.176,11621.138
9,2010,139.971150,134.00,0.343766,641,531,82.763846,100.000000,30.269928,178.525674,8.058474,21.263999,2.722590,12.526958,0.132473,-5.971,24988.444
